# LeetCode 1512 — Number of Good Pairs bằng Python

**Link đề:** https://leetcode.com/problems/number-of-good-pairs/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Math, Counting (đếm)

```python
class Solution:
    def numIdenticalPairs(self, nums: list[int]) -> int:
        ...
```

### Mục tiêu học
1. Hiểu "cặp tốt" `(i, j)` với `i < j` và `nums[i] == nums[j]`; liệt kê bằng tay.
2. Viết brute force hai vòng lặp lồng nhau và đếm được số phép so sánh `n(n-1)/2`.
3. Dùng `dict` / `collections.Counter` để **đếm tần suất**, rồi áp dụng công thức tổ hợp `k(k-1)/2`.
4. Nắm lời giải **một lần duyệt**: mỗi phần tử mới tạo thêm đúng `count[x]` cặp với các bản sao đã gặp.
5. Trace, chứng minh, test đối chiếu; chuyển kỹ thuật sang biến thể (cặp có hiệu `k`, cặp bằng nhau trong khoảng cách…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `nums`, trả về số **cặp tốt** (good pairs).

Cặp chỉ số `(i, j)` là **tốt** nếu `nums[i] == nums[j]` **và** `i < j`.

| `nums` | Output | Các cặp tốt (0-based) |
|---|---:|---|
| `[1,2,3,1,1,3]` | 4 | `(0,3), (0,4), (3,4), (2,5)` |
| `[1,1,1,1]` | 6 | mọi cặp: `(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)` |
| `[1,2,3]` | 0 | không có giá trị lặp |

**Constraints:** `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

Chú ý: điều kiện `i < j` nghĩa là mỗi cặp chỉ đếm **một lần** (không đếm cả `(3,0)` lẫn `(0,3)`) và không ghép phần tử với **chính nó** (`i ≠ j`).

In [ ]:
nums = [1, 2, 3, 1, 1, 3]
for i, x in enumerate(nums):
    print(f"index {i}: {x}")

Nhìn bảng: giá trị `1` ở các chỉ số `0, 3, 4`; giá trị `3` ở `2, 5`; giá trị `2` chỉ ở `1`.
- Từ ba chỉ số của `1`, chọn 2 → `(0,3), (0,4), (3,4)` = 3 cặp.
- Từ hai chỉ số của `3` → `(2,5)` = 1 cặp.
- Tổng 4 ✔.

## 2. Nền tảng Python cần biết

### 2.1. Hai vòng lặp lồng nhau với `j` bắt đầu từ `i + 1`

`for i in range(n): for j in range(i + 1, n)` sinh **mọi cặp** `i < j`, mỗi cặp đúng một lần.

In [ ]:
n = 4
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((i, j))
print(pairs)
print("số cặp:", len(pairs), "= n(n-1)/2 =", n * (n - 1) // 2)

### 2.2. `dict` đếm tần suất

Đếm số lần xuất hiện của mỗi giá trị:
```python
count = {}
for x in nums:
    count[x] = count.get(x, 0) + 1
```
`count.get(x, 0)` trả `count[x]` nếu đã có, ngược lại trả `0` (không lỗi `KeyError`).

In [ ]:
count = {}
for x in [1, 2, 3, 1, 1, 3]:
    count[x] = count.get(x, 0) + 1
    print(f"gặp {x} -> count = {count}")

### 2.3. `collections.Counter` — dict chuyên đếm

`Counter(iterable)` đếm sẵn; truy cập khóa chưa có trả `0` thay vì lỗi. `.values()` cho các tần suất, `.most_common()` sắp theo tần suất giảm dần.

In [ ]:
from collections import Counter
c = Counter([1, 2, 3, 1, 1, 3])
print(c)
print("c[1] =", c[1], "| c[99] =", c[99])
print("values:", list(c.values()))
print("most_common:", c.most_common())

### 2.4. Tổ hợp chập 2: `k(k-1)/2`

Có `k` vị trí cùng giá trị, số cách chọn **2 vị trí khác nhau không kể thứ tự** là

```
C(k, 2) = k * (k - 1) / 2
```

Lý do: chọn vị trí thứ nhất `k` cách, thứ hai `k - 1` cách → `k(k-1)` cặp **có thứ tự**; mỗi cặp không thứ tự bị đếm 2 lần (`(a,b)` và `(b,a)`) → chia 2.

Python có `math.comb(k, 2)` tính sẵn. Dùng `//` để giữ kết quả là `int` (`k(k-1)` luôn chẵn nên chia hết).

In [ ]:
import math
for k in range(0, 6):
    print(f"k={k}: k*(k-1)//2 = {k * (k - 1) // 2} | math.comb(k, 2) = {math.comb(k, 2)}")

## 3. Cách 1 — Brute force: thử mọi cặp

Dịch thẳng định nghĩa.

In [ ]:
def num_identical_pairs_bruteforce(nums: list[int]) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):      # đảm bảo i < j
            if nums[i] == nums[j]:
                count += 1
    return count

print(num_identical_pairs_bruteforce([1, 2, 3, 1, 1, 3]))   # 4
print(num_identical_pairs_bruteforce([1, 1, 1, 1]))         # 6
print(num_identical_pairs_bruteforce([1, 2, 3]))            # 0

### Code nộp LeetCode — brute force

Với `n <= 100`: tối đa `4950` phép so sánh → Accepted.

In [ ]:
class Solution:
    def numIdenticalPairs(self, nums: list[int]) -> int:
        n = len(nums)
        return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] == nums[j])

print(Solution().numIdenticalPairs([1, 2, 3, 1, 1, 3]))

Dòng `sum(1 for ... for ... if ...)` là **generator expression** lồng hai `for` — đọc từ trái sang phải giống hai vòng lặp lồng nhau; mỗi cặp thỏa điều kiện đóng góp `1` vào tổng.

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

## 4. Nút thắt — ta không cần biết *cặp nào*, chỉ cần *bao nhiêu*

Brute force so sánh cả những cặp khác giá trị (vô ích). Thật ra đáp án chỉ phụ thuộc vào **mỗi giá trị xuất hiện bao nhiêu lần**:

- `[1, 2, 3, 1, 1, 3]` và `[3, 1, 1, 2, 3, 1]` có cùng tần suất `{1: 3, 2: 1, 3: 2}` → cùng đáp án.
- Vị trí cụ thể không quan trọng — vì từ bất kỳ 2 vị trí khác nhau, luôn sắp được thành `i < j` đúng một cách.

In [ ]:
a = [1, 2, 3, 1, 1, 3]
b2 = [3, 1, 1, 2, 3, 1]
print(Counter(a) == Counter(b2), num_identical_pairs_bruteforce(a), num_identical_pairs_bruteforce(b2))

## 5. Cách 2 — Đếm tần suất rồi cộng `k(k-1)/2` (gợi ý của đề)

In [ ]:
def num_identical_pairs_formula(nums: list[int]) -> int:
    freq = Counter(nums)
    total = 0
    for k in freq.values():
        total += k * (k - 1) // 2
    return total

print(num_identical_pairs_formula([1, 2, 3, 1, 1, 3]))   # 3*2/2 + 1*0/2 + 2*1/2 = 3 + 0 + 1 = 4
print(num_identical_pairs_formula([1, 1, 1, 1]))         # 4*3/2 = 6
print(num_identical_pairs_formula([1, 2, 3]))            # 0

## 6. Insight — Cách 3: một lần duyệt, "mỗi phần tử mới bắt cặp với các bản sao đã gặp"

Duyệt từ trái sang phải. Khi đứng ở chỉ số `j` với giá trị `x`, những cặp tốt **có `j` là đầu phải** chính là các `i < j` mà `nums[i] == x`. Số lượng `i` như vậy = **số lần `x` đã xuất hiện trước đó** = `count[x]` (trước khi tăng).

Vậy:
```python
ans += count[x]     # j tạo thêm count[x] cặp mới
count[x] += 1       # ghi nhận x vừa xuất hiện
```

Thứ tự **cộng trước, tăng sau** là then chốt (giống Two Sum: kiểm tra trước, lưu sau) — để không ghép `x` với chính nó.

Liên hệ với công thức: với giá trị xuất hiện `k` lần, lần lượt cộng `0 + 1 + 2 + ... + (k-1) = k(k-1)/2` — đúng công thức cách 2!

In [ ]:
print("0+1+2+3 =", 0 + 1 + 2 + 3, "| k=4: k(k-1)/2 =", 4 * 3 // 2)

In [ ]:
def num_identical_pairs_one_pass(nums: list[int]) -> int:
    count = {}
    ans = 0
    for x in nums:
        ans += count.get(x, 0)          # cặp mới với các bản sao đã gặp
        count[x] = count.get(x, 0) + 1  # sau đó mới ghi nhận x
    return ans

print(num_identical_pairs_one_pass([1, 2, 3, 1, 1, 3]))
print(num_identical_pairs_one_pass([1, 1, 1, 1]))
print(num_identical_pairs_one_pass([1, 2, 3]))

## 7. TRACE từng bước — cách 3

Ví dụ `nums = [1, 2, 3, 1, 1, 3]`:

| j | x | count[x] trước | cặp mới | ans | count sau |
|---:|---:|---:|---|---:|---|
| 0 | 1 | 0 | – | 0 | {1:1} |
| 1 | 2 | 0 | – | 0 | {1:1, 2:1} |
| 2 | 3 | 0 | – | 0 | {1:1, 2:1, 3:1} |
| 3 | 1 | 1 | (0,3) | 1 | {1:2, 2:1, 3:1} |
| 4 | 1 | 2 | (0,4), (3,4) | 3 | {1:3, 2:1, 3:1} |
| 5 | 3 | 1 | (2,5) | 4 | {1:3, 2:1, 3:2} |

Kết quả `4` ✔. Hàm trace dưới đây còn lưu danh sách chỉ số của mỗi giá trị để **in ra chính các cặp mới**.

In [ ]:
def trace_good_pairs(nums):
    positions = {}            # giá trị -> list các chỉ số đã gặp (chỉ để in minh họa)
    ans = 0
    for j, x in enumerate(nums):
        before = positions.get(x, [])
        new_pairs = [(i, j) for i in before]
        ans += len(new_pairs)
        positions[x] = before + [j]
        print(f"j={j}, x={x}: đã gặp {x} tại {before} -> cặp mới {new_pairs} -> ans = {ans}")
    print("Tổng số cặp tốt:", ans)
    return ans

trace_good_pairs([1, 2, 3, 1, 1, 3])

Trace trường hợp "toàn bộ giống nhau" để thấy `0 + 1 + 2 + 3`:

In [ ]:
trace_good_pairs([1, 1, 1, 1])

## 8. Cách 4 — Mảng đếm thay cho dict (tận dụng `1 <= nums[i] <= 100`)

Khi giá trị nằm trong khoảng nhỏ đã biết, có thể dùng list `cnt = [0] * 101` làm "dict" với khóa là chỉ số. Truy cập list nhanh và không cần `get`.

In [ ]:
def num_identical_pairs_array(nums: list[int]) -> int:
    cnt = [0] * 101            # chỉ số 0..100
    ans = 0
    for x in nums:
        ans += cnt[x]
        cnt[x] += 1
    return ans

print(num_identical_pairs_array([1, 2, 3, 1, 1, 3]), num_identical_pairs_array([100, 100]))

## 9. Vì sao đúng?

**Cách 2:** Phân hoạch tập các cặp tốt theo **giá trị chung** `v`. Cặp tốt có giá trị `v` ⇔ chọn 2 vị trí khác nhau trong `k_v` vị trí mang giá trị `v`, rồi sắp thành `i < j` (đúng 1 cách). Số cách = `C(k_v, 2)`. Các nhóm rời nhau → cộng lại.

**Cách 3 — invariant:** trước khi xử lý chỉ số `j`:
- `count[v]` = số chỉ số `i < j` có `nums[i] == v`;
- `ans` = số cặp tốt `(i, j')` với `j' < j`.

Ở bước `j`, các cặp tốt có đầu phải đúng bằng `j` là `(i, j)` với `i < j`, `nums[i] == nums[j]` — có đúng `count[nums[j]]` cặp. Cộng vào `ans`, rồi tăng `count[nums[j]]` → invariant đúng cho `j + 1`. Mỗi cặp tốt được đếm **đúng một lần** (tại đầu phải `j` của nó).

## 10. Code nộp LeetCode — một lần duyệt

In [ ]:
class Solution:
    def numIdenticalPairs(self, nums: list[int]) -> int:
        count = {}
        ans = 0
        for x in nums:
            ans += count.get(x, 0)
            count[x] = count.get(x, 0) + 1
        return ans

sol = Solution()
assert sol.numIdenticalPairs([1, 2, 3, 1, 1, 3]) == 4
assert sol.numIdenticalPairs([1, 1, 1, 1]) == 6
assert sol.numIdenticalPairs([1, 2, 3]) == 0
print("PASS 3 ví dụ đề")

Bản công thức một dòng (cũng Accepted):
```python
from collections import Counter
class Solution:
    def numIdenticalPairs(self, nums: list[int]) -> int:
        return sum(k * (k - 1) // 2 for k in Counter(nums).values())
```

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `m` = số giá trị phân biệt (`m <= min(n, 100)`).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` | dịch thẳng định nghĩa |
| 2. Counter + `k(k-1)/2` | `O(n + m)` = `O(n)` | `O(m)` | dùng công thức tổ hợp |
| 3. Một lần duyệt + dict | `O(n)` trung bình | `O(m)` | mẫu "đếm cặp khi duyệt" |
| 4. Mảng đếm 101 ô | `O(n)` | `O(101)` = `O(1)` | chỉ khi miền giá trị nhỏ |

"Trung bình" với dict vì tra cứu băm `O(1)` trung bình. Đếm số phép so sánh của brute force để cảm nhận:

In [ ]:
for n in [10, 100, 1000]:
    print(f"n={n:>4}: brute force so sánh {n * (n - 1) // 2:>7} cặp | một lần duyệt: {n} bước")

## 12. Kiểm thử: mẫu đề, biên, đối chiếu

In [ ]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<28} -> {actual}")

tests = [
    ("đề 1",                      [1, 2, 3, 1, 1, 3], 4),
    ("đề 2 (toàn bằng nhau)",     [1, 1, 1, 1],       6),
    ("đề 3 (phân biệt)",          [1, 2, 3],          0),
    ("1 phần tử",                 [5],                0),
    ("2 phần tử bằng",            [7, 7],             1),
    ("hai nhóm",                  [2, 2, 3, 3],       2),
    ("100 phần tử giống nhau",    [9] * 100,          4950),
    ("giá trị biên 1 và 100",     [1, 100, 1, 100],   2),
]
fs = (num_identical_pairs_bruteforce, num_identical_pairs_formula, num_identical_pairs_one_pass, num_identical_pairs_array)
for name, arr, exp in tests:
    for f in fs:
        assert f(arr) == exp, (f.__name__, arr)
    check(name, Solution().numIdenticalPairs(arr), exp)

In [ ]:
import random
random.seed(2026)
for _ in range(2000):
    arr = [random.randint(1, random.choice([3, 10, 100])) for _ in range(random.randint(1, 100))]
    exp = num_identical_pairs_bruteforce(arr)
    for f in fs[1:]:
        assert f(arr) == exp, (f.__name__, arr)
    assert sum(math.comb(k, 2) for k in Counter(arr).values()) == exp
print("2000 mảng ngẫu nhiên: 4 cách + math.comb khớp nhau. PASS")

## 13. Lỗi người mới hay gặp

### Lỗi 1: `j` chạy từ `0` → đếm cả `(i, i)` và đếm mỗi cặp 2 lần
```python
for i in range(n):
    for j in range(n):          # SAI
        if nums[i] == nums[j]:
            count += 1
```
`[1, 1, 1, 1]` cho `16` thay vì `6` (= 4 cặp `(i,i)` + 12 cặp có thứ tự).

### Lỗi 2: Một lần duyệt nhưng **tăng trước, cộng sau**
```python
count[x] = count.get(x, 0) + 1
ans += count[x]                 # SAI: tự ghép với chính mình
```

### Lỗi 3: Dùng `k * (k - 1) / 2` (chia thực) → kết quả là `float` (`4.0`); LeetCode yêu cầu `int`.

### Lỗi 4: Cộng `k * k // 2` hoặc `k // 2` (nhầm công thức)
Với `k = 3`: `k*k//2 = 4`, `k//2 = 1`, nhưng đúng là `3`.

### Lỗi 5: `count[x] += 1` khi `x` chưa có trong dict thường → `KeyError`. Dùng `get`, `Counter` hoặc `defaultdict(int)`.

In [ ]:
def wrong_full_loop(nums):
    n, c = len(nums), 0
    for i in range(n):
        for j in range(n):
            if nums[i] == nums[j]:
                c += 1
    return c

def wrong_increment_first(nums):
    count, ans = {}, 0
    for x in nums:
        count[x] = count.get(x, 0) + 1
        ans += count[x]
    return ans

arr = [1, 1, 1, 1]
print("đúng                 :", num_identical_pairs_one_pass(arr))
print("lỗi 1 j từ 0         :", wrong_full_loop(arr), "-> (16 - 4) / 2 =", (wrong_full_loop(arr) - len(arr)) // 2)
print("lỗi 2 tăng trước     :", wrong_increment_first(arr))
print("lỗi 3 chia thực      :", sum(k * (k - 1) / 2 for k in Counter(arr).values()))
print("lỗi 4 k*k//2 với k=3 :", 3 * 3 // 2, "| đúng:", 3 * 2 // 2)
try:
    d = {}
    d[5] += 1
except KeyError as e:
    print("lỗi 5 KeyError:", e)
from collections import defaultdict
dd = defaultdict(int)
dd[5] += 1
print("defaultdict(int) ổn:", dict(dd))

`defaultdict(int)` tự tạo giá trị mặc định `int() = 0` cho khóa mới, nên `dd[x] += 1` không lỗi.

Nhận xét ở lỗi 1: kết quả sai vẫn "sửa" được bằng `(tổng - n) / 2` — đúng là ý tưởng tổ hợp: bỏ các cặp `(i,i)` rồi chia 2 vì mỗi cặp bị đếm hai chiều.

## 14. Mẹo nhận dạng pattern

- "Đếm số cặp `(i < j)` thỏa `f(nums[i], nums[j])`" và điều kiện chỉ phụ thuộc **giá trị** → **hash map đếm** khi duyệt: tại `j`, cộng số phần tử trước đó "khớp" với `nums[j]`.
- Điều kiện "bằng nhau" → nhóm theo giá trị, dùng `C(k, 2)`.
- Điều kiện "tổng bằng `t`" / "hiệu bằng `k`" → cùng khuôn: `ans += count[t - x]` hoặc `count[x - k] + count[x + k]`.

Họ bài: 1 Two Sum, 2006 Count Number of Pairs With Absolute Difference K, 1711 Count Good Meals, 2364 Count Number of Bad Pairs, 1128 Number of Equivalent Domino Pairs.

## 15. Biến thể

### Biến thể A — Cặp có hiệu tuyệt đối bằng `k` (LeetCode 2006)

In [ ]:
def count_k_difference(nums, k):
    count, ans = {}, 0
    for x in nums:
        ans += count.get(x - k, 0) + count.get(x + k, 0)
        count[x] = count.get(x, 0) + 1
    return ans

print(count_k_difference([1, 2, 2, 1], 1))   # 4
print(count_k_difference([3, 2, 1, 5, 4], 2))   # 3

Chú ý: với `k = 0`, công thức trên đếm `count[x]` **hai lần** — cần xử lý riêng (bài này giả sử `k >= 1`).

### Biến thể B — Domino tương đương (LeetCode 1128)
Domino `[a, b]` và `[c, d]` tương đương nếu `(a == c and b == d)` hoặc `(a == d and b == c)`. Chuẩn hóa mỗi domino thành tuple **đã sắp** `(min, max)` → quay về bài Good Pairs.

In [ ]:
def num_equiv_domino_pairs(dominoes):
    count, ans = {}, 0
    for a, b3 in dominoes:
        key = (min(a, b3), max(a, b3))
        ans += count.get(key, 0)
        count[key] = count.get(key, 0) + 1
    return ans

print(num_equiv_domino_pairs([[1, 2], [2, 1], [3, 4], [5, 6]]))   # 1

## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
`nums = [4, 4, 2, 4, 2, 7, 4]`. Đếm tay số cặp tốt bằng công thức `k(k-1)/2`.

In [ ]:
def exercise_by_hand() -> int:
    # TODO: tần suất của 4? của 2? của 7?
    pass

# assert exercise_by_hand() == num_identical_pairs_bruteforce([4, 4, 2, 4, 2, 7, 4])

### Bài 2 (điền chỗ trống — một lần duyệt)

In [ ]:
def exercise_good_pairs(nums: list[int]) -> int:
    count = {}
    ans = 0
    for x in nums:
        # TODO 1: cộng số lần x đã xuất hiện vào ans
        # TODO 2: tăng count[x]
        pass
    return ans

# assert exercise_good_pairs([1, 2, 3, 1, 1, 3]) == 4
# assert exercise_good_pairs([1, 1, 1, 1]) == 6

### Bài 3 (liệt kê cặp)
Trả về **danh sách** các cặp tốt `(i, j)` theo thứ tự `j` tăng, cùng `j` thì `i` tăng.

In [ ]:
def exercise_list_pairs(nums: list[int]) -> list[tuple[int, int]]:
    # TODO
    pass

# assert exercise_list_pairs([1, 2, 3, 1, 1, 3]) == [(0, 3), (0, 4), (3, 4), (2, 5)]

### Bài 4 (Count Bad Pairs — LeetCode 2364, phiên bản đơn giản)
Cặp `(i, j)` với `i < j` là **xấu** nếu `nums[i] != nums[j]`. Đếm số cặp xấu trong `O(n)`.

Gợi ý: tổng số cặp − số cặp tốt.

In [ ]:
def exercise_bad_pairs(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_bad_pairs([1, 2, 3, 1, 1, 3]) == 11
# assert exercise_bad_pairs([5, 5, 5]) == 0

### Bài 5 (cặp có tổng bằng `t`)
Đếm số cặp `i < j` mà `nums[i] + nums[j] == t`, trong `O(n)`.

In [ ]:
def exercise_pairs_with_sum(nums: list[int], t: int) -> int:
    # TODO
    pass

# assert exercise_pairs_with_sum([1, 5, 7, -1, 5], 6) == 3
# assert exercise_pairs_with_sum([1, 1, 1, 1], 2) == 6

### Bài 6 (mini challenge — cặp tốt gần nhau)
Đếm số cặp `i < j` với `nums[i] == nums[j]` **và** `j - i <= d`.

Ví dụ `nums = [1, 2, 1, 1, 2]`, `d = 2`: các cặp bằng nhau là `(0,2)`, `(0,3)`, `(2,3)`, `(1,4)`. Trong đó `(0,3)` và `(1,4)` có khoảng cách `3 > 2` nên bị loại. Đáp án `2`.

In [ ]:
def exercise_close_good_pairs(nums: list[int], d: int) -> int:
    # TODO: gợi ý — cửa sổ trượt chứa d phần tử gần nhất bên trái + dict đếm
    pass

# assert exercise_close_good_pairs([1, 2, 1, 1, 2], 2) == 2
# assert exercise_close_good_pairs([1, 1, 1, 1], 1) == 3

### Gợi ý ba mức

**Bài 1** — M1: đếm tần suất. M2: `4` xuất hiện 4 lần, `2` hai lần, `7` một lần. M3: `6 + 1 + 0`.

**Bài 2** — M1: cộng trước, tăng sau. M2: `ans += count.get(x, 0)`. M3: `count[x] = count.get(x, 0) + 1`.

**Bài 3** — M1: lưu **danh sách chỉ số** cho mỗi giá trị thay vì chỉ đếm. M2: `positions = {}`; tại `j`, mọi `i` trong `positions[x]` tạo cặp `(i, j)`. M3: `res.extend((i, j) for i in positions.get(x, []))` rồi `positions.setdefault(x, []).append(j)`.

**Bài 4** — M1: tổng số cặp `i < j` là `n(n-1)/2`. M2: trừ đi số cặp tốt. M3: `n * (n - 1) // 2 - exercise_good_pairs(nums)`.

**Bài 5** — M1: tại `j`, cần các `i` trước đó có `nums[i] = t - nums[j]`. M2: `ans += count.get(t - x, 0)`. M3: sau đó `count[x] += 1` (cộng trước, tăng sau).

**Bài 6** — M1: dict chỉ chứa các phần tử trong cửa sổ `[j - d, j - 1]`. M2: trước khi xử lý `j`, nếu `j - d - 1 >= 0` thì giảm `count[nums[j - d - 1]]`. M3: rồi `ans += count.get(x, 0)`, `count[x] += 1`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [ ]:
def answer_by_hand():
    return 4 * 3 // 2 + 2 * 1 // 2 + 0     # 6 + 1 + 0 = 7

def answer_good_pairs(nums):
    count, ans = {}, 0
    for x in nums:
        ans += count.get(x, 0)
        count[x] = count.get(x, 0) + 1
    return ans

def answer_list_pairs(nums):
    positions, res = {}, []
    for j, x in enumerate(nums):
        res.extend((i, j) for i in positions.get(x, []))
        positions.setdefault(x, []).append(j)
    return res

def answer_bad_pairs(nums):
    n = len(nums)
    return n * (n - 1) // 2 - answer_good_pairs(nums)

def answer_pairs_with_sum(nums, t):
    count, ans = {}, 0
    for x in nums:
        ans += count.get(t - x, 0)
        count[x] = count.get(x, 0) + 1
    return ans

def answer_close_good_pairs(nums, d):
    count, ans = {}, 0
    for j, x in enumerate(nums):
        out = j - d - 1                       # phần tử vừa rời khỏi cửa sổ
        if out >= 0:
            count[nums[out]] -= 1
        ans += count.get(x, 0)
        count[x] = count.get(x, 0) + 1
    return ans

assert answer_by_hand() == num_identical_pairs_bruteforce([4, 4, 2, 4, 2, 7, 4]) == 7
assert answer_good_pairs([1, 2, 3, 1, 1, 3]) == 4 and answer_good_pairs([1, 1, 1, 1]) == 6
assert answer_list_pairs([1, 2, 3, 1, 1, 3]) == [(0, 3), (0, 4), (3, 4), (2, 5)]
assert answer_bad_pairs([1, 2, 3, 1, 1, 3]) == 11 and answer_bad_pairs([5, 5, 5]) == 0
assert answer_pairs_with_sum([1, 5, 7, -1, 5], 6) == 3
assert answer_pairs_with_sum([1, 1, 1, 1], 2) == 6
assert answer_close_good_pairs([1, 2, 1, 1, 2], 2) == 2
assert answer_close_good_pairs([1, 1, 1, 1], 1) == 3
for _ in range(500):
    arr = [random.randint(1, 4) for _ in range(random.randint(1, 15))]
    d = random.randint(1, 6)
    brute = sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] == arr[j] and j - i <= d)
    assert answer_close_good_pairs(arr, d) == brute, (arr, d)
    assert len(answer_list_pairs(arr)) == answer_good_pairs(arr)
print("Tất cả đáp án PASS!")

`dict.setdefault(key, mặc_định)` trả `dict[key]` nếu có; nếu chưa có thì gán `dict[key] = mặc_định` rồi trả về nó — tiện để `.append` vào list của khóa mới.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numIdenticalPairs(self, nums: list[int]) -> int:
        count, ans = {}, 0
        for x in nums:
            ans += count.get(x, 0)          # cộng TRƯỚC: cặp với bản sao đã gặp
            count[x] = count.get(x, 0) + 1  # tăng SAU
        return ans
```

Công thức: `sum(k * (k - 1) // 2 for k in Counter(nums).values())`.

**Checklist trước khi submit:**
- [ ] Chỉ đếm `i < j` (không đếm `(i, i)`, không đếm hai chiều)?
- [ ] Một lần duyệt: **cộng trước, tăng sau**?
- [ ] Dùng `//` để kết quả là `int`?
- [ ] Công thức đúng là `k(k-1)/2`?
- [ ] Không `KeyError` (dùng `get` / `Counter` / `defaultdict`)?